# BOARDROOM — GRPO Training (Colab)

Train a language model to play **BOARDROOM**, a 7-company corporate warfare game, using GRPO reinforcement learning via TRL + Unsloth 4-bit QLoRA.

Two modes available:
- **Single-agent** (`train_grpo.py`) — model plays Vermillion Capital vs 6 random heuristic opponents. Fast baseline.
- **Self-play** (`train_grpo_selfplay.py`) — all 7 companies use the model, primary rotates each episode. Real competitive signal.

**Before running:** Runtime → Change runtime type → **T4 GPU**

---
**Live environment:** https://huggingface.co/spaces/nothr/boardroom  
**Best model:** https://huggingface.co/nothr/boardroom-grpo-lora-L2-best  
**Training logs (W&B):** https://api.wandb.ai/links/me-harshithreddy-gitam/xnb97tdo

In [ ]:
# Cell 1: Verify GPU
!nvidia-smi
import torch
print(f'CUDA: {torch.cuda.is_available()}  |  GPU: {torch.cuda.get_device_name(0)}')
print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

In [ ]:
# Cell 2: Install deps
# Unsloth first — pins specific torch/triton versions
!pip install -q unsloth

!pip install -q \
    "trl>=0.27.0" \
    "peft>=0.13.0" \
    "accelerate>=0.34.0" \
    "datasets>=3.0.0" \
    "bitsandbytes>=0.44.0" \
    "wandb>=0.18.0" \
    "matplotlib>=3.8.0" \
    "openenv-core[core]>=0.2.2"

print('Done. Restart runtime if prompted, then continue from Cell 3.')

In [ ]:
# Cell 3: Clone repo + (optional) mount Drive for persistent checkpoints
import os

REPO_DIR = "/content/boardroom"

if not os.path.exists(REPO_DIR):
    !git clone https://huggingface.co/spaces/nothr/boardroom {REPO_DIR}
else:
    !git -C {REPO_DIR} pull

USE_DRIVE = False   # set True to persist checkpoints across sessions
if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    CHECKPOINT_DIR = "/content/drive/MyDrive/boardroom-checkpoints"
else:
    CHECKPOINT_DIR = "/content/checkpoints"

os.makedirs(CHECKPOINT_DIR, exist_ok=True)
print(f'Repo:        {REPO_DIR}')
print(f'Checkpoints: {CHECKPOINT_DIR}')

In [ ]:
# Cell 4: W&B login (optional — skip if you don't want experiment tracking)
import wandb
wandb.login()   # paste key from wandb.ai/settings when prompted
print('W&B ready')

---
## Choose training mode

Run **either** Cell 5a (single-agent, faster) **or** Cell 5b (self-play, real signal). Not both.

In [ ]:
# Cell 5a: Single-agent training
# Model plays Vermillion Capital vs 6 random heuristic opponents.
# ~30 min / 200 steps on T4. Good for testing env wiring, not real strategy.

import sys, argparse
sys.path.insert(0, REPO_DIR)

import train.train_grpo as tg

args = argparse.Namespace(
    model_id           = "google/gemma-4-E2B-it",
    output_dir         = CHECKPOINT_DIR,
    run_name           = "boardroom-grpo-single",
    max_steps          = 200,
    batch_size         = 4,
    num_generations    = 4,
    max_completion_len = 128,
    warmup_steps       = 10,
    refresh_every      = 50,
    n_rollout_episodes = 50,
    fp16               = True,   # T4 uses fp16; set False on A100/L40S
    hub_model_id       = None,   # e.g. "yourusername/boardroom-lora" to push best
)

tg.parse_args = lambda: args
tg.main()

In [ ]:
# Cell 5b: Self-play training  ← RECOMMENDED
# All 7 companies use the same model. Primary company rotates each episode.
# Model learns general strategy, not just how to beat random opponents.
# ~7x more generation per step — expect ~3-4 hrs / 200 steps on T4.
# Use L40S on HF Jobs for faster runs.

import sys, argparse
sys.path.insert(0, REPO_DIR)

import train.train_grpo_selfplay as sp

args = argparse.Namespace(
    model_id           = "google/gemma-4-E2B-it",
    output_dir         = CHECKPOINT_DIR,
    run_name           = "boardroom-selfplay-v1",
    max_steps          = 200,
    batch_size         = 4,
    num_generations    = 4,
    max_completion_len = 128,
    warmup_steps       = 10,
    refresh_every      = 50,
    n_seeds            = 50,
    fp16               = True,
    hub_model_id       = None,   # e.g. "yourusername/boardroom-grpo-selfplay"
)

sp.parse_args = lambda: args
sp.main()

In [ ]:
# Cell 6: Show training plots inline
from IPython.display import Image, display
from pathlib import Path

for png in ['loss.png', 'reward.png']:
    p = Path(CHECKPOINT_DIR) / png
    if p.exists():
        print(f'── {png} ──')
        display(Image(str(p)))
    else:
        print(f'{png} not found yet')

In [ ]:
# Cell 7: Print training summary
import json
from pathlib import Path

summary_path = Path(CHECKPOINT_DIR) / 'summary.json'
if summary_path.exists():
    s = json.loads(summary_path.read_text())
    print(json.dumps(s, indent=2))
else:
    print('summary.json not found — training may still be running')

In [ ]:
# Cell 8: Evaluate trained model vs heuristic baseline (20 episodes)
# Uses the best adapter saved during training, or falls back to the published model.

import subprocess, sys

ADAPTER = str(Path(CHECKPOINT_DIR) / 'best_adapter')
HUB_MODEL = "nothr/boardroom-grpo-lora-L2-best"  # fallback if local adapter missing

cmd = [
    sys.executable, f"{REPO_DIR}/scripts/eval_grpo.py",
    "--hub-model-id", HUB_MODEL,
    "--n-episodes", "20",
    "--no-wandb",
]

result = subprocess.run(cmd, capture_output=False, text=True)
if result.returncode != 0:
    print('Eval failed — check output above')

In [ ]:
# Cell 9: Pull and display W&B training metrics (requires wandb login from Cell 4)
# Replace RUN_PATH with your own run path from wandb.ai

import wandb
import pandas as pd
import matplotlib.pyplot as plt

RUN_PATH = "me-harshithreddy-gitam/boardroom/runs/tpcf64f1"  # published run

api = wandb.Api()
run = api.run(RUN_PATH)
df  = run.history()

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.suptitle('BOARDROOM GRPO Training', fontsize=13)

for ax, col, color, title in [
    (axes[0], 'train/reward',   '#5ca8e0', 'Reward'),
    (axes[1], 'train/loss',     '#e05c5c', 'Loss'),
    (axes[2], 'train/kl',       '#5ce09a', 'KL Divergence'),
]:
    d = df[['_step', col]].dropna()
    ax.plot(d['_step'], d[col], color=color)
    ax.set_title(title); ax.set_xlabel('Step'); ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('/content/training_curves.png', dpi=150, bbox_inches='tight')
plt.show()

r = df['train/reward'].dropna()
print(f'Reward — mean: {r.mean():.4f}  max: {r.max():.4f}  final: {r.iloc[-1]:.4f}')
print(f'Steps logged: {len(df)}')